[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-ames.ipynb)

# Full Project: House Price Prediction (Ames, Iowa)

*AIBits Academy · Machine Learning End To End · Full Project*

A classic 2,930-row, 82-feature regression benchmark — systematic missing-value handling, feature engineering, and a genuine 7-model comparison won by XGBoost.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['AmesHousing.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> A buyer, seller, or lender wants a defensible estimate of a house's fair market price from its raw characteristics — lot size, quality ratings, basement/garage details, sale conditions — rather than relying on a single agent's opinion. This is the same class of problem as the Municipal Bond and Employee Flight Risk projects elsewhere in this course: turning many raw, messy columns into one trustworthy number.

> **Dataset**
>
> **2,930 records, 82 raw features**, covering residential sales in Ames, Iowa (2006–2010), introduced by De Cock (2011). Prices are in USD. This dataset is US in origin, so it is presented in its native USD/Ames context rather than reframed with Indian pricing — consistent with how the Municipal Bond and Employee Attrition projects elsewhere in this course are handled.

## Step 1 — Systematic Missing-Value Handling

82 raw features meant checking every column for missingness rather than guessing. The columns with the most missing values, and the documented reason behind each:

| Column | Missing | % | Fill strategy (per data dictionary) |
|---|---|---|---|
| Pool QC | 2,917 | 99.56% | "No Pool" — verified via Pool Area = 0 for all 2,917 |
| Misc Feature | 2,824 | 96.38% | "No feature" — verified via Misc Val = 0 for the same rows |
| Alley | 2,732 | 93.24% | "No Alley" |
| Fence | 2,358 | 80.48% | "No Fence" |
| Fireplace Qu | 1,422 | 48.53% | "No Fireplace" |
| Lot Frontage | 490 | 16.72% | 0 (not connected to a street) |
| Garage (5 cols) | 157–159 | ~5.4% | "No Garage" / 0 — two edge-case rows needed manual cross-checking against `Garage Cars` |
| Basement (7 cols) | 1–83 | ≤2.8% | "No Basement" / 0 — a handful of rows had partial basement data requiring row-by-row inspection |
| Mas Vnr Area/Type | 23 | 0.78% | 0 / "None" |
| Electrical | 1 | 0.03% | Mode imputation (the only column without a documented "none" category) |

The key discipline demonstrated here: **Pool QC and Misc Feature look like candidates for outright deletion at 99.56% and 96.38% missing** — the instinctive move on the Data Preprocessing page's threshold logic. But cross-checking against `Pool Area` and `Misc Val` (both genuinely 0 for the same rows) confirms the missingness isn't random data loss at all — it's the data dictionary's way of encoding "this house doesn't have one." Deleting these columns would have silently discarded real, meaningful information.

In [ ]:
import pandas as pd

df = pd.read_csv('AmesHousing.csv')
# Verify the "missing = doesn't have one" hypothesis before filling
print(df['Pool Area'].value_counts().head(1))   # 2917 rows with Pool Area == 0

no_feature = {'Pool QC': 'No Pool', 'Misc Feature': 'No feature', 'Alley': 'No Alley', 'Fence': 'No Fence',
              'Fireplace Qu': 'No Fireplace', 'Mas Vnr Type': 'None',
              'Garage Qual': 'No Garage', 'Garage Cond': 'No Garage', 'Garage Finish': 'No Garage', 'Garage Type': 'No Garage',
              'Bsmt Exposure': 'No Basement', 'BsmtFin Type 2': 'No Basement', 'Bsmt Cond': 'No Basement',
              'Bsmt Qual': 'No Basement', 'BsmtFin Type 1': 'No Basement'}
for col, fill in no_feature.items():
    df[col] = df[col].fillna(fill)
zero_fill = ['Lot Frontage', 'Garage Yr Blt', 'Mas Vnr Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'BsmtFin SF 1',
             'BsmtFin SF 2', 'Total Bsmt SF', 'Bsmt Unf SF', 'Garage Area', 'Garage Cars']
df[zero_fill] = df[zero_fill].fillna(0)
df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])
print("Remaining missing values:", df.isna().values.sum())

## Step 2 — Outlier Removal

Plotting `SalePrice` against `Gr Liv Area` (above-ground living area) surfaces exactly 5 unusual points — large houses that sold for unexpectedly low prices, a pattern the dataset's original author (De Cock, 2011) explicitly flags and recommends removing:

| Row | Gr Liv Area | Sale Type | Sale Condition | SalePrice |
|---|---|---|---|---|
| 1498 | 5,642 | New | Partial | $160,000 |
| 1760 | 4,476 | WD | Abnorml | $745,000 |
| 1767 | 4,316 | WD | Normal | $755,000 |
| 2180 | 5,095 | New | Partial | $183,850 |
| 2181 | 4,676 | New | Partial | $184,750 |

Rows 1760/1767 are the opposite anomaly — enormous houses selling for very high prices consistent with their size, but so far outside the bulk of the distribution they still distort a linear fit. All 5 rows are removed by keeping only `Gr Liv Area < 4000`, then the index is reset.

## Step 3 — Feature Engineering

Two engineering moves, guided directly by the correlation heatmap:

- **Polynomial & interaction features:** `Overall Qual` and `Gr Liv Area` showed the strongest correlation with `SalePrice`, so squared and cubed versions of each were added, plus an `OverallQual_GrLivArea` product term.
- **Multicollinearity removal:** `Garage Cars` and `Garage Area` were near-perfectly correlated (more car capacity almost always means more area), as were `Gr Liv Area` and `TotRms AbvGrd`. One feature from each pair (`Garage Cars`, `TotRms AbvGrd`) was dropped.
- **Ordinal encoding** for 18 quality-rated columns (`Exter Qual`, `Bsmt Qual`, `Kitchen Qual`, `Functional`, `Garage Finish`, etc.) using the data dictionary's explicit Ex>Gd>TA>Fa>Po ranking, rather than one-hot encoding, since these genuinely have an order.
- **One-hot encoding** for the remaining nominal categoricals via `pd.get_dummies()`, bringing the final feature count to **242**.

## Step 4 — Comparing 7 Regression Techniques

Each model was tuned with `GridSearchCV`/`RandomizedSearchCV` (3-fold CV, scoring on negative MAE), then scored once on a held-out 25% test set:

| Model | Test MAE (USD) |
|---|---|
| **XGBoost** | **$12,556.68** |
| Support Vector Regression (SVR) | $12,874.93 |
| Random Forest | $14,506.46 |
| Elastic Net | $14,767.91 |
| Ridge | $15,270.46 |
| Decision Tree | $20,873.95 |
| K-Nearest Neighbors | $22,780.14 |

For context, the training target `SalePrice` has a mean of $179,846.69 and a median of $159,895 — so XGBoost's $12,556.68 MAE represents roughly 7% average error relative to a typical house price, on data with a right-skewed range from $12,789 to $625,000.

## Visualizing the 7-Model Comparison

Lower is better here — XGBoost (green) and SVR (purple, the "strong second place" from the callout below) both clear the field by a wide margin over the remaining five models.

## Step 5 — Feature Importance: What Actually Drives Price?

XGBoost and Random Forest agree on the broad strokes but rank them differently — XGBoost's top features are dominated by raw size/age measures (`Lot Area`, `Total Bsmt SF`, `Year Built`, `Gr Liv Area`, `1st Flr SF`), while Random Forest weights the engineered `OverallQual_GrLivArea` interaction term and the polynomial `Overall Qual` features far more heavily — direct evidence that the feature-engineering step in Step 3 wasn't wasted effort for at least one of the two model families.

> **💡 SVR's Strong Second Place Is a Useful Surprise**
>
> Support Vector Regression outperforming Random Forest and every linear model here is a reminder that a properly-tuned kernel method (covered on the Support Vector Machines and Kernel Methods Beyond SVM pages) is a genuinely competitive choice for structured regression problems, not merely a classification tool — provided features are scaled first, which every pipeline in this comparison did via `StandardScaler`.

## Key Business Takeaways

- A high missingness percentage (Pool QC at 99.56%) is not automatically a "delete this column" signal — cross-checking against a related column (`Pool Area`) revealed the missingness was meaningful, not random.
- Removing just 5 rows (0.17% of the dataset) based on a domain-expert-flagged outlier pattern measurably stabilizes a linear fit, without needing any exotic robust-regression technique.
- The best model (XGBoost) and the runner-up (SVR) belong to entirely different model families — a reminder that "which one model family wins" is dataset-specific, and comparing several is worth the extra compute on any project where the final choice matters.

## Practice Questions

## Steps 2–4 in code (compact version)

The lesson describes outlier removal, feature engineering and a seven-model comparison in prose and tables. Here is a compact, runnable version: remove the five documented outliers, add the two engineered features, then compare three models with a fixed 75/25 split. (The lesson's table comes from a longer search over seven tuned models, so absolute numbers here are a little different.)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Step 2: the five documented outliers all have Gr Liv Area >= 4000
df = df[df['Gr Liv Area'] < 4000].reset_index(drop=True)
print("after outlier removal:", df.shape)

# Step 3: two engineered features
df['OverallQual_GrLivArea'] = df['Overall Qual'] * df['Gr Liv Area']
df['TotalSF'] = df['Total Bsmt SF'] + df['1st Flr SF'] + df['2nd Flr SF']

# Step 4: one-hot encode text columns, split 75/25, compare models by mean absolute error
X = pd.get_dummies(df.drop(columns=['SalePrice', 'PID', 'Order']), drop_first=True).astype(float)
y = df['SalePrice']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

models = {
    'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=30)),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.8, random_state=42),
}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    print(f"{name:14s} test MAE = ${mean_absolute_error(y_te, m.predict(X_te)):,.0f}")

### Which features drive price?

The same question the lesson asks in Step 5, answered with the XGBoost model above.

In [ ]:
imp = pd.Series(models['XGBoost'].feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(8).round(3))

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · How many houses have no pool?

Store in `no_pool_share` the fraction of houses whose `Pool Area` is 0 (using the current `df`).

In [ ]:
no_pool_share = None   # TODO


In [ ]:
try:
    check("about 99.5%", 0.99 < no_pool_share < 1.0)
    check("exact", abs(no_pool_share - (df["Pool Area"] == 0).mean()) < 1e-12)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
no_pool_share = float((df["Pool Area"] == 0).mean())

```

</details>

### Exercise 2 · Medium · Age matters

Engineer `HouseAge` = `Yr Sold` − `Year Built` and store its correlation with `SalePrice` in `corr_age`. It should be clearly negative (older houses sell for less).

In [ ]:
corr_age = None   # TODO


In [ ]:
try:
    check("negative correlation", corr_age < -0.4)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
df["HouseAge"] = df["Yr Sold"] - df["Year Built"]
corr_age = float(df["HouseAge"].corr(df["SalePrice"]))

```

</details>

### Exercise 3 · Stretch · Tame the skew

`SalePrice` is right-skewed. Store its skewness in `skew_raw` and the skewness of `np.log(SalePrice)` in `skew_log`. The log version should be much closer to zero.

In [ ]:
skew_raw = skew_log = None   # TODO


In [ ]:
try:
    check("raw is skewed", skew_raw > 1)
    check("log is far more symmetric", abs(skew_log) < 0.5 * abs(skew_raw))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
skew_raw = float(df["SalePrice"].skew())
skew_log = float(np.log(df["SalePrice"]).skew())

```

Log-transforming a skewed target often makes linear models fit far better - and turns errors into percentages.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: House Price Prediction (Ames, Iowa)**.*